# 자연어 생성

## Sequence to Sequence (Seq2Seq)

- 자연어 처리에서 기존의 일반적인 판별 모델(NLU / RNN, LSTM...)은 문장 내 단어들을 각 Node가 처리하여 하나의 결과를 계산
- 입력 Sequence를 이용해, 하나의 나열된 Sequence를 출력하는 구조를 만들어 학습을 수행
- 기계 번역 / 챗봇 / 문장 요약 / STT
- Seq2Seq 구조 (3가지 파트 / 두 개의 RNN 구조)
    1. Encoder
        - Input Sequence(입력 문장)를 받아 해당 문장을 특정 Vector로 변환하는 구조
        - 입력값의 정보를 각 Node가 받아서 순차적으로 처리 (RNN 계열)
        - 여기서 처리된 입력 문장의 정보를 Context Vector 파트로 보냄
    2. Context Vector
        - Encoder에 의해 하나의 문장 정보가 처리된 Fixed-Size Vector (고정된 벡터 / 입력 문장을 정해진 사이즈의 벡터로 압축)
        - Context Vector를 Decoder로 보내어, 입력 문장에 맞는 출력을 Decoder가 수행할 수 있도록 정보를 전달하는 역할
        - 문장이 길어질수록 정보 손실이 발생하는 한계점이 존재
    3. Decoder
        - Output Sequence를 학습하고, Input Sequence에 해당하는 적절한 Output을 출력

- 작동 순서
    1. 입력 문장(Input Sequence)을 단어 토큰화를 통해 단어 단위로 분할
        - ※ 이 노트북은 단어가 아니라 **글자 단위**로 분할 (영어 67자, 한국어 802자)
    2. 각 토큰을 문장 순서대로 Encoder에 있는 RNN 모형에 입력 → RNN 모형이 입력 문장을 학습
    3. Encoder의 마지막 Node에서 계산된 값을 Context Vector로 변환 → Decoder로 전달
        - ※ 코드에서는 인코더 LSTM의 마지막 상태 [state_h, state_c]를 그대로 Context Vector로 사용
    4. Decoder는 앞서 받은 Context Vector를 RNN 첫 번째 Node에 전달
        - ※ 코드에서는 디코더 LSTM의 initial_state(초기 상태)로 넣어줌
    5. Decoder는 문장의 시작을 의미하는 심볼인 SOS(Start Of Sequence)를 RNN(D) 첫 번째 Node에 넣어 순차적으로 처리
        - ※ 이 노트북에서 SOS = '\t'
    6. 이후 Decoder는 문장을 이어받으며 다음에 등장할 확률이 높은 단어를 예측하며 문장을 생성
        - ※ 학습 시: 다음 Node의 입력으로 **정답 단어**를 넣음 (Teacher Forcing)
        - ※ 예측 시: 다음 Node의 입력으로 **직전에 예측한 단어**를 넣음
    7. EOS(End Of Sequence) 토큰이 등장하거나 문장의 최대 길이에 도달할 때까지 5~6번 과정을 반복하며 단어를 학습 / 예측
        - ※ 이 노트북에서 EOS = '\n'
    8. Decoder는 SOS에서 시작해 EOS가 나올 때까지 학습을 수행하며, 예측된 값은 Output Sequence로 출력

    
- 한계
    - 입력 문장이 길수록 Context Vector에 정보를 압축할 때 손상이 발생
    - Context Vector가 고정 크기라서 문장의 모든 정보를 담기 어려움

![Seq2Seq 개념](images/S1_Seq2Seq_concept.webp)

![Seq2Seq 모형 구조 - 시간축으로 펼쳐 보기](images/S2_Seq2Seq_unrolled.webp)

- 그림의 ①~⑧ = 위 작동 순서 1~8번
- 인코더: LSTM(64, return_state=True) → 중간 출력은 버리고(×), 마지막 [h, c]만 컨텍스트로 전달
- 디코더: LSTM(64, return_sequences=True, return_state=True) → 모든 시점에서 글자 출력
- 디코더 입력과 출력은 한 칸씩 밀려 있음 (입력 \t 들 어 와 . → 출력 들 어 와 . \n)
- 각 시점 출력: Dense(802, softmax) → argmax로 확률이 가장 높은 글자 선택

- Decoder : 출력될 글자를 학습하고 생성하는 영역
    - Teacher Forcing : 정답을 디코더의 입력으로 사용
        - 학습할 때 정답 문장을 디코더의 입력으로 한 칸씩 밀어서 입력
        - 특정 위치에서 오답이 발생해도, 해당 시점의 손실만 높아지므로 다음 입력에 영향을 주지 않음
        - 학습이 안정적이고 빠르며, 학습 초반의 엉터리 예측이 뒤로 번지지 않아서 모든 시점이 올바른 앞부분을 보고 학습
        - 입력 문장 전체가 미리 준비되어 있어, 학습할 때 모든 시점의 손실을 한 번에 같이 계산
        - ※ 코드: decoder_input_data(\t 들 어 와 .) → decoder_target_data(들 어 와 . \n), 한 칸 차이
    - Autoregressive (자기 회귀) : 모델이 방금 예측한 글자를 다시 입력하여 추론에 사용
        - 추론 때는 정답이 없으니, 직전 예측을 다음 입력으로 넣고 한 글자씩 반복하여 예측
        - ※ 코드: decode_sequence 함수의 while 반복문 ('\n'이 나오거나 최대 길이까지)
    - 디코더는 학습용 모형과 추론용 모형을 따로 구성해야 함 (가중치는 동일하게 공유)
        - ※ 코드: 학습용 model / 추론용 encoder_model, decoder_model → 같은 LSTM·Dense 층을 재사용
    - 구조적 간극 발생 / 노출 편향(Exposure Bias)이 발생
    - 노출 편향(Exposure Bias) : 학습 땐 항상 정답의 앞부분만 보다가, 추론할 땐 틀릴 수 있는 자신의 예측값을 보기 때문에 발생하는 불일치 현상
        - ※ 추론 중 한 글자를 틀리면, 그 틀린 글자가 다음 입력이 되어 오류가 뒤로 번질 수 있음

![Seq2Seq 학습 vs 추론, 한계](images/S3_Seq2Seq_train_vs_inference.webp)

- 학습(Teacher Forcing): decoder_input[t+1] = decoder_target[t] → 정답이 한 칸 밀려 다음 입력이 됨
    - 틀려도 다음 입력은 정답 → 그 시점 손실만 커짐, 5개 시점을 한 번에 계산
- 추론(자기회귀): 직전 예측 글자를 다음 입력으로 → while 루프로 한 글자씩, '\n'이 나오면 멈춤
    - 한 번 틀리면 뒤로 오류가 이어짐 = 노출 편향(exposure bias)
- 한계 ① 고정 크기 컨텍스트 = 병목: 8글자든 26글자든 [h, c] 128개로 압축
- 한계 ② 기울기 소실/폭주: 역전파 신호가 곱해지며 앞으로 갈수록 약해짐(또는 폭주) → 문장 앞부분 학습이 어려움
- 해결 → Attention: 디코더가 글자를 만들 때마다 인코더의 모든 시점 출력을 다시 보고, 필요한 위치에 가중치를 줌
    - 매 시점 새 컨텍스트 c_t 생성 → 병목 해소, 먼 거리도 한 번에 연결
    - 코드: 인코더를 return_sequences=True로 바꿔 모든 시점 출력을 남겨둠

In [ ]:
# 텍스트 파일 호출 / 영어 문장이 입력되었을 때 한국어 문장으로 번역하는 모형을 생성


with open('../data/08_Translation_Data/kor.txt', encoding='UTF-8') as file:
    text1 = file.read().split('\n')


# → my_code에서 한 칸 위(Day10)로 올라가 data 폴더의 kor.txt를 UTF-8로 열겠다
# → 파일 전체를 읽은 뒤 줄바꿈(\n) 기준으로 잘라 한 줄씩 리스트에 담겠다

In [ ]:
# 영어/한국어 문장과 글자 목록을 담을 그릇 만들기


input_text = []   # 영어 문장을 리스트로 선언
target_text = []  # 한국어 문장을 리스트로 선언

# 영어/한국어 글자가 숫자로 변환된 정보를 처리하는 Set 구조를 선언
# 영어 -> 알파벳 (i'm fine -> i/m/f/i/n/e -> 1/2/3/1/4/5)
# 한국어 -> 글자 (안녕하세요 -> 안/녕/하/세/요 -> 1/2/3/4/5)
input_char_list = set()
target_char_list = set()


# → 영어 문장, 한국어 문장을 담을 빈 리스트를 만들겠다
# → 등장한 글자를 중복 없이 모을 빈 set을 만들겠다 (같은 글자 = 같은 번호)

In [ ]:
# 데이터 구조 확인


print(len(text1))                  # 전체 문장 수 확인
print(text1[10].split('\t'))       # 한 줄 = 영어 / 한국어 / 출처 3조각
print(text1[1005].split('\t')[0])  # 영어만 추출
print(text1[1005].split('\t')[1])  # 한국어만 추출


# → 전체 줄 수를 확인하겠다 (5891)
# → 10번째 줄을 탭 기준으로 잘라 3조각 구조를 확인하겠다
# → 1005번째 줄에서 0번(영어) / 1번(한국어)만 꺼내겠다 ('Is this a radio?' / '이것은 라디오입니까?')

5891
['Jump!', '점프!', 'CC-BY 2.0 (France) Attribution: tatoeba.org #1102981 (jamessilver) & #8355893 (Eunhee)']
Is this a radio?
이것은 라디오입니까?


In [ ]:
# 반복문을 이용하여, 3000개 문장에 대해서만 영어/한국어를 나누어 자료구조로 선언


for line in text1[:3000]:
    # Decoder에서 처리될 한국어 문장의 경우
    # SOS <Start Of Sequence>와 EOS <End Of Sequence>를 부착하여 자료구조에 추가
    # '\t' -> <SOS> // '\n' -> <EOS>
    try:
        input_txt, target_txt, ect = line.split('\t')
        target_txt_seq = '\t' + target_txt + '\n'
        # 영어 문장과 한국어 문장을 각 리스트에 추가 
        input_text.append(input_txt)
        target_text.append(target_txt_seq)
        
    # 아래는 앞서 try 구문에서 오류가 발생할 경우 , 오류 메세지를 출력하는 코드 
    except Exception as e:
        print(e)
        continue
    
    # 각 영어/한국어 문장을 알파벳과 한글자 단위로 잘라 Set 구조에 추가 
    # 영어 : Come on ! on me ! -> c/o/m/e/ /o/n/!/ /o/n/ /m/e/! -> c,o,m,e,n,! 
    # 한국어 : 빨리 ! 어서와! -> 빨/리/!/ 어/서/와/!/ -> 빨,리,어,서,와,! 
    for x in input_txt:
        input_char_list.add(x)
    for x2 in target_txt_seq:
        target_char_list.add(x2)


# → 앞에서 3,000줄만 꺼내 한 줄씩 반복하겠다
# → 탭 기준으로 영어 / 한국어 / 출처(ect) 3개로 나눠 받겠다
# → 디코더가 쓸 한국어 문장 앞에 \t(SOS), 뒤에 \n(EOS)를 붙이겠다
# → 영어는 input_text에, 신호 붙인 한국어는 target_text에 추가하겠다
# → 형식이 안 맞는 줄은 에러만 출력하고 다음 줄로 넘어가겠다
# → 영어/한국어 문장을 한 글자씩 돌며 각각의 글자 set에 추가하겠다
# ※ 이 셀은 한 번만 실행 (두 번 실행하면 문장이 중복으로 쌓임 → 위 그릇 셀부터 다시 실행)

In [ ]:
input_char_list


{' ',
 '!',
 '"',
 "'",
 ',',
 '-',
 '.',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 ':',
 '?',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'Y',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z'}

In [ ]:
# Text to Sequence 사전을 만들기 위해, 각 글자를 정렬한 뒤 리스트 구조로 선언 -> 토큰의 개수를 확인


input_characters = sorted(list(input_char_list))
target_characters = sorted(list(target_char_list))

print(len(input_characters), len(target_characters))
print(target_characters[:10])


# → set은 순서가 없으니 리스트로 바꾼 뒤 정렬해서, 글자 순서를 고정하겠다
# → 영어 글자 종류 수(67), 한국어 글자 종류 수(802)를 확인하겠다
# → 한국어 글자 목록의 앞 10개를 보겠다

67 802
['\t', '\n', ' ', '!', '"', '(', ')', ',', '.', '0']


In [ ]:
 len(input_characters)

67

In [ ]:
len(target_characters)

802

In [ ]:
# 각 토큰(알파벳과 한글 글자 조합)에 숫자를 부여 / Corpus 구성
# Eng2Num / Kor2Num
# enumerate() -> 특정 리스트에 있는 객체를 하나씩 가져와 숫자를 부여하는 함수


input_token_index = dict([(y, x) for x, y in enumerate(input_characters)])
target_token_index = dict([(y, x) for x, y in enumerate(target_characters)])


# → 정렬된 글자 목록을 enumerate로 돌며 (번호, 글자)를 꺼내겠다
# → 순서를 (글자, 번호)로 뒤집어 {글자: 번호} 사전을 만들겠다

In [ ]:
target_token_index

{'\t': 0,
 '\n': 1,
 ' ': 2,
 '!': 3,
 '"': 4,
 '(': 5,
 ')': 6,
 ',': 7,
 '.': 8,
 '0': 9,
 '1': 10,
 '2': 11,
 '3': 12,
 '4': 13,
 '5': 14,
 '6': 15,
 '7': 16,
 ':': 17,
 '?': 18,
 'T': 19,
 'V': 20,
 '~': 21,
 '가': 22,
 '각': 23,
 '간': 24,
 '갇': 25,
 '갈': 26,
 '감': 27,
 '갑': 28,
 '값': 29,
 '갔': 30,
 '강': 31,
 '갖': 32,
 '같': 33,
 '갛': 34,
 '개': 35,
 '갰': 36,
 '걀': 37,
 '걔': 38,
 '거': 39,
 '걱': 40,
 '건': 41,
 '걷': 42,
 '걸': 43,
 '검': 44,
 '겁': 45,
 '것': 46,
 '게': 47,
 '겐': 48,
 '겠': 49,
 '겨': 50,
 '격': 51,
 '견': 52,
 '결': 53,
 '겼': 54,
 '경': 55,
 '곁': 56,
 '계': 57,
 '고': 58,
 '곤': 59,
 '곧': 60,
 '골': 61,
 '곱': 62,
 '곳': 63,
 '공': 64,
 '과': 65,
 '관': 66,
 '괜': 67,
 '괴': 68,
 '굉': 69,
 '교': 70,
 '구': 71,
 '국': 72,
 '군': 73,
 '굴': 74,
 '굶': 75,
 '굽': 76,
 '궁': 77,
 '권': 78,
 '귀': 79,
 '그': 80,
 '극': 81,
 '근': 82,
 '글': 83,
 '금': 84,
 '급': 85,
 '긋': 86,
 '긍': 87,
 '기': 88,
 '긴': 89,
 '길': 90,
 '김': 91,
 '깊': 92,
 '까': 93,
 '깎': 94,
 '깐': 95,
 '깔': 96,
 '깜': 97,
 '깡': 98,
 '깨': 99,
 '꺼': 10

In [ ]:
import numpy as np

In [ ]:
# Padding을 수행함과 동시에 Text -> Matrix로 변환
# 각 언어(영어 / 한국어) 문장에서 가장 문장 길이가 긴 문장의 글자 수를 확인
# 원-핫 인코딩을 담을 빈 3차원 배열(전부 0) 생성


# 영어/한국어 문장을 한 문장씩 x로 가져와, 문장별 글자 수를 세어 리스트로 만들고 -> max로 가장 큰 값을 확인
max_encoder_seq_length = max([len(x) for x in input_text])
max_decoder_seq_length = max([len(x) for x in target_text])


# 토큰을 넣을 Matrix의 구조를 구성 (비어 있는 구조)
# np.zeros(모양, dtype) -> 지정한 모양의 배열을 만들고 모든 칸을 0으로 채우는 함수
#   ① 문장 개수          -> 몇 개의 문장인지
#   ② 가장 긴 문장 글자 수 -> 한 문장에 칸이 몇 줄인지
#   ③ 글자 종류 수        -> 글자 하나를 몇 칸짜리 원-핫으로 표현할지
#   dtype='float32'      -> 각 칸의 숫자를 32비트 소수로 저장 (딥러닝 계산용, 메모리 절약)

# Input 문장이 들어갈 Matrix (인코더 입력 / 영어)
encoder_input_data = np.zeros(
    (len(input_text), max_encoder_seq_length, len(input_characters)),
    dtype='float32')

# Target 문장이 들어갈 Matrix (학습 과정 : 디코더 입력 / Teacher Forcing -> \t 들 어 와 .)
decoder_input_data = np.zeros(
    (len(input_text), max_decoder_seq_length, len(target_characters)),
    dtype='float32')

# Target 문장이 예측될 Matrix (학습 과정 : 예측값과 비교할 정답 라벨 -> 들 어 와 . \n)
decoder_target_data = np.zeros(
    (len(input_text), max_decoder_seq_length, len(target_characters)),
    dtype='float32')

print(max_encoder_seq_length, max_decoder_seq_length)
print(encoder_input_data.shape, decoder_input_data.shape, decoder_target_data.shape)


# → 영어 문장 중 가장 긴 길이(26), 한국어 문장(SOS·EOS 포함) 중 가장 긴 길이(30)를 구하겠다
# → (문장 수, 최대 글자 수, 글자 종류 수) 모양의 0으로 가득 찬 배열을 3개 만들겠다
# → 인코더 입력 (3000, 26, 67) = 3000페이지 × 26줄 × 67칸짜리 빈 모눈 공책
# → 디코더 입력·정답 (3000, 30, 802), 영어-한국어가 짝지어진 문장이라 ① 자리는 둘 다 3000
# → 디코더 정답은 입력보다 한 칸 밀린 내용을 담을 예정 (Teacher Forcing)
# → 가장 긴 문장에 칸을 맞추고, 짧은 문장은 남는 칸을 0으로 두는 것이 Padding
# ※ 세 배열 모두 학습(model.fit)에 사용 / 추론 때는 정답 없이 직전 예측을 다음 입력으로 사용

26 30
(3000, 26, 67) (3000, 30, 802) (3000, 30, 802)


In [ ]:
encoder_input_data.shape # (문장 줄 , 각 문장 내 글자 위치 (토큰 최대 길이), 각 글자의 번호)

(3000, 26, 67)

In [ ]:
# 구성된 Matrix에 각 글자에 매칭되는 값을 입력 (원-핫 인코딩 : 해당 글자 칸만 1로 변경)
#
# [예시] 사전 {a: 0, b: 1}, 문장 'aba' → 한 문장 = (글자 위치 3줄 × 글자 종류 2칸) 표
#            a칸  b칸
#   1번째 줄 [ 1,   0 ]   ← 1번째 글자 a → a칸에 1
#   2번째 줄 [ 0,   1 ]   ← 2번째 글자 b → b칸에 1
#   3번째 줄 [ 1,   0 ]   ← 3번째 글자 a → a칸에 1
# → 이런 표가 문장 수만큼 쌓이면 3차원 Matrix : [문장 번호 i, 글자 위치 k(j), 글자 번호]


# 3000쌍의 영어-한국어 문장을 하나씩 꺼내며 반복
#   zip(input_text, target_text) → 같은 순서의 영어·한국어 문장을 짝지음 ('Come in.', '\t들어와.\n')
#   enumerate(...)               → 짝마다 번호 i를 붙임 (i = 몇 번째 문장 = 3차원 배열의 몇 번째 페이지)
for i, (input_txt, target_txt) in enumerate(zip(input_text, target_text)):

    # 영어 문장을 한 글자씩 꺼내며 반복 (k = 문장 안에서 몇 번째 글자인지 = 몇 번째 줄)
    for k, char in enumerate(input_txt):
        # [i번 문장, k번째 줄, 이 글자의 번호 칸] 한 곳만 1로 칠함
        #   input_token_index[char] → 영어 사전에서 이 글자의 번호를 찾아옴 (예: 'C' → 20)
        encoder_input_data[i, k, input_token_index[char]] = 1

    # 한국어 문장을 한 글자씩 꺼내며 반복 (j = 문장 안에서 몇 번째 글자인지 = 몇 번째 줄)
    for j, char in enumerate(target_txt):
        # 디코더 입력 : [i번 문장, j번째 줄, 이 글자의 번호 칸]을 1로 칠함 (\t 들 어 와 . \n 그대로)
        decoder_input_data[i, j, target_token_index[char]] = 1

        # SOS에 대한 처리 : j=0인 첫 글자 \t는 "시작 신호"일 뿐 맞혀야 할 정답이 아니므로 건너뜀
        if j > 0:
            # 디코더 정답 : 같은 글자를 한 줄 앞(j-1)에 칠함 → 정답이 입력보다 한 칸 앞서게 됨
            #   입력 0줄 '\t' ↔ 정답 0줄 '들' / 입력 1줄 '들' ↔ 정답 1줄 '어' ... (Teacher Forcing)
            decoder_target_data[i, j-1, target_token_index[char]] = 1


# 1로 칠해진 칸의 개수를 세어 제대로 채워졌는지 확인
print(encoder_input_data.sum())                           # 54497 = 3000개 영어 문장의 전체 글자 수
print(decoder_input_data.sum(), decoder_target_data.sum())  # 정답 쪽이 3000 적음 = 문장마다 \t 하나씩 제외


# → zip으로 영어-한국어 문장 짝을 묶고, enumerate로 문장 번호 i를 붙여 3000쌍을 하나씩 꺼내겠다
# → 영어 문장의 k번째 글자를 사전에서 번호로 바꿔, [i번 문장, k번째 줄, 글자 번호 칸]을 1로 칠하겠다
# → 한국어 문장의 j번째 글자도 같은 방식으로 디코더 입력에 칠하겠다
# → 같은 글자를 디코더 정답에는 한 줄 앞(j-1)에 칠하겠다 → 정답이 입력보다 한 칸 앞서게 됨
# → j=0인 첫 글자 \t(SOS)는 정답에 넣지 않겠다 (정답은 '들'부터 시작)
# ※ 값을 1로 "덮어쓰는" 코드라서, 여러 번 실행해도 결과가 같음 (append와 달리 중복 걱정 없음)
# ※ 여기까지가 데이터 전처리(벡터화) 단계 : 문장(Text) → 모델에 넣을 3차원 숫자 배열(Matrix)

54497.0
38814.0 35814.0


In [ ]:
encoder_input_data.sum()

np.float32(54497.0)

## Seq2Seq 모형구성
``

In [ ]:
# 학습 하이퍼파라미터 설정


batch_size = 32  # 훈련 배치 사이즈 (실질적 출력이 나오려면 : 128 이상)
epochs = 30      # Weight update를 위한 훈련 반복 수 (500회 이상)
node_num = 64    # 각 RNN 모형에 있는 Node 수 (1024개 이상)


# → 한 번에 32문장씩 묶어서 학습하겠다 (batch_size)
# → 전체 3000문장을 30바퀴 반복해서 학습하겠다 (epochs)
# → 인코더·디코더 LSTM의 노드(기억 칸)를 64개로 하겠다 → 컨텍스트 벡터 = 64 + 64 = 128개 숫자
# ※ 괄호 안 숫자는 번역이 실제로 쓸 만해지려면 필요한 규모 → 수업에서는 빠르게 돌려보려고 작게 설정

In [ ]:
# 모델 구성에 필요한 Keras 부품 불러오기


from keras.models import Model
from keras.layers import Input
from keras.layers import LSTM
from keras.layers import Dense


# → Model : 입력과 출력을 연결해 하나의 모델로 묶는 틀을 불러오겠다
# → Input : 모델에 들어올 데이터의 모양을 알려주는 입구를 불러오겠다
# → LSTM  : 인코더·디코더로 쓸 순환 신경망 층을 불러오겠다
# → Dense : 디코더 출력을 802개 글자 확률로 바꿀 층을 불러오겠다
# ※ 커널을 켜고 처음 실행할 때 1~3분 걸릴 수 있음 → 멈추지 말고 기다리기

In [ ]:
# Encoder 구성 : 영어 문장을 읽고 마지막 상태 [h, c]를 컨텍스트 벡터로 만든다
# 영어 문장이 들어가 LSTM 모델에 의해 학습이 수행되도록 구성


encoder_input = Input(shape=(None, len(input_characters)))  # 영어 문장(원-핫 Matrix)이 들어오는 Layer
encoder = LSTM(node_num, return_state=True)

# Encoder LSTM 모델에 Input Data를 입력해 출력을 계산
# encoder_output : LSTM 모델에 의해 최종적으로 계산된 값
# state_h : Hidden State (마지막 시점, 즉 마지막 글자까지 읽은 뒤의 단기 기억)
# state_c : Cell State (마지막 시점의 장기 기억)
encoder_output, state_h, state_c = encoder(encoder_input)

# Encoder가 입력을 처리한 후, 얻은 정보를 Decoder로 전달
encoder_state = [state_h, state_c]  # Context Vector


# → 인코더 입구 : (글자 수는 자유, 글자 하나는 67칸 원-핫) 모양의 데이터를 받겠다
# → 노드 64개짜리 LSTM을 만들고, 마지막 기억 상태(h, c)도 돌려받도록 설정하겠다
# → 입구 데이터를 LSTM에 통과시켜 출력, h, c 세 가지를 받겠다
# → h와 c를 묶어 컨텍스트 벡터(encoder_state, 64 + 64 = 128개 숫자)로 만들겠다 → 디코더의 초기 상태로 넘길 예정

In [ ]:
# Decoder 구성 (Teacher Forcing)
# 앞서 Encoder가 처리한 정보를 받아, 한국어 문장이 들어가 LSTM 모형에 의해 학습이 되도록 구성


decoder_input = Input(shape=(None, len(target_characters)))  # 학습해야 할 한국어 문장(원-핫 Matrix)이 들어오는 Layer
decoder = LSTM(node_num, return_sequences=True, return_state=True)
# return_sequences=True : 각 시점(글자 위치)마다 출력이 나오도록 구성

# initial_state=encoder_state : 앞서 처리된 Encoder의 Context Vector를 Decoder의 초기 상태로 지정
# decoder_output : 모든 시점의 출력
# deco_h, deco_c : 마지막 시점의 상태 → 학습에선 안 쓰고, 추론용 모델에서 다음 글자로 넘길 때 사용
decoder_output, deco_h, deco_c = decoder(decoder_input, initial_state=encoder_state)

# Softmax 함수에 의해 출력을 계산 → 매 시점 출력을 802개 한국어 글자의 확률로 변환
decoder_dense = Dense(len(target_characters), activation='softmax')
decoder_output = decoder_dense(decoder_output)


# → 디코더 입구 : (글자 수 자유, 글자 하나는 802칸 원-핫) 모양의 한국어 입력을 받겠다
# → 노드 64개짜리 LSTM을 만들되, 모든 시점의 출력과 마지막 상태를 둘 다 돌려받겠다
# → 인코더의 컨텍스트 벡터를 디코더 LSTM의 초기 기억으로 넣고 입력을 통과시키겠다 (작동 순서 4번)
# → 매 시점 출력(64개 숫자)을 802개 글자 확률로 바꾸는 Dense(softmax) 층을 만들겠다
# → LSTM 출력을 Dense에 통과시켜 최종 예측(시점마다 802개 확률)을 얻겠다
# ※ decoder, decoder_dense 층은 나중에 추론용 모델에서 그대로 재사용 (가중치 공유)

In [38]:
# Seq2Seq 모형을 학습 (Encoder - 영어 문장 이해 / Decoder - 한국어 문장 생성)


# Model(입력, 출력) : 따로 만든 층들을 입구와 출구로 연결해 하나의 모델로 묶음
#   [encoder_input, decoder_input] → 입구 2개 (영어 원-핫 / 한 칸 밀린 한국어 원-핫)
#   decoder_output                 → 출구 1개 (시점마다 802개 한국어 글자 확률)
#   두 입구는 decoder의 initial_state=encoder_state로 이미 안에서 이어져 있음
model = Model([encoder_input, decoder_input], decoder_output)

# compile : 학습 규칙을 정함 (아직 학습은 시작하지 않음)
#   optimizer='adam'                  → 오차를 줄이기 위해 가중치를 고치는 방법 (가장 널리 쓰는 방식)
#   loss='categorical_crossentropy'   → 예측이 얼마나 틀렸는지 재는 방법 = −log P(정답 글자)
#                                        802개 글자 중 하나를 고르는 "분류" 문제라서 사용
model.compile(optimizer='adam', loss='categorical_crossentropy')

# fit : 실제 학습 시작 (Teacher Forcing)
#   [encoder_input_data, decoder_input_data] → 모델의 입구 2개에 넣을 데이터
#                                               영어 (3000, 26, 67) / 한국어 입력 \t 들 어 와 . (3000, 30, 802)
#   decoder_target_data  → 예측값과 비교할 정답 라벨 (들 어 와 . \n), 입력보다 한 칸 앞선 내용
#   batch_size=32        → 32문장씩 묶어서 계산하고 한 번씩 가중치를 고침
#   epochs=30            → 전체 학습 데이터를 30바퀴 반복
#   validation_split=0.2 → 뒤쪽 20%(600문장)는 학습에 쓰지 않고, 매 바퀴 끝에 "처음 보는 문장" 검사용으로 사용
model.fit([encoder_input_data, decoder_input_data], decoder_target_data,
          batch_size=batch_size, epochs=epochs, validation_split=0.2)


# → 입력 2개(영어 입구, 한국어 입구)와 출력 1개(글자 확률)를 연결해 하나의 모델로 묶겠다
# → 가중치를 고치는 방법은 adam, 오차 계산은 categorical_crossentropy로 설정하겠다
# → 영어·한국어(한 칸 밀린) 원-핫 배열을 넣고, 한국어 정답 배열과 비교하며 학습하겠다
# → 32문장씩 묶어서 30바퀴 반복하고, 데이터의 20%는 학습에 쓰지 않고 검증용으로 떼어 두겠다
# ※ 출력의 loss = 학습 문장 오차 / val_loss = 검증 문장(처음 보는 문장) 오차
# ※ loss는 계속 내려가는데 val_loss가 멈추거나 오르면 과적합 (학습 문장을 외워 버린 상태)

Epoch 1/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 22s 160ms/step - loss: 1.9992 - val_loss: 2.3354
Epoch 2/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 116ms/step - loss: 1.7194 - val_loss: 2.3344
Epoch 3/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 14s 155ms/step - loss: 1.7080 - val_loss: 2.3187
Epoch 4/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 114ms/step - loss: 1.6978 - val_loss: 2.3043
Epoch 5/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 119ms/step - loss: 1.6882 - val_loss: 2.2980
Epoch 6/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 118ms/step - loss: 1.6807 - val_loss: 2.2896
Epoch 7/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 8s 103ms/step - loss: 1.6731 - val_loss: 2.2764
Epoch 8/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 85ms/step - loss: 1.6647 - val_loss: 2.2652
Epoch 9/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 85ms/step - loss: 1.6574 - val_loss: 2.2703
Epoch 10/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 83ms/step - loss: 1.6510 - val_loss: 2.2465
Epoch 11/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 85ms/step - loss: 1.6434 - val_loss: 2.2392
Epoch 12/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 120ms/step -

In [39]:
# ① 모델 정의 (입력과 출력 연결)


model = Model( [encoder_input, decoder_input] , decoder_output )


# → Model(입력, 출력) : 케라스(Keras)에서 "이 입력이 들어가면 이 출력이 나온다"고 연결해 하나의 모델로 만들겠다
# → [encoder_input, decoder_input] : 입력을 2개 받겠다
#     - encoder_input : 영어 문장이 들어가는 입구
#     - decoder_input : 한국어 문장이 들어가는 입구 (학습 때 정답 문장을 넣어줌 → 아래 Teacher Forcing 참고)
# → decoder_output : 디코더가 단어(또는 글자)마다 "다음에 올 것이 무엇일 확률"을 내놓는 것을 최종 출력으로 삼겠다
# → 즉, 앞 셀들에서 따로 만든 인코더와 디코더 부품을 하나로 조립하겠다

In [40]:
# ② 컴파일 (학습 방법 설정)


model.compile(optimizer='adam', loss='categorical_crossentropy')


# → compile : 학습을 시작하기 전에 "어떻게 학습할지" 규칙을 정하겠다
# → optimizer='adam'
#     - 옵티마이저(optimizer) : 틀린 만큼 모델 내부 값(가중치)을 어느 방향으로 얼마나 고칠지 정하는 방법
#     - adam : 가장 널리 쓰이는 옵티마이저로, 고치는 보폭을 상황에 맞춰 자동 조절해주는 방식을 쓰겠다
# → loss='categorical_crossentropy'
#     - 손실 함수(loss) : 모델의 예측이 정답과 얼마나 다른지를 숫자로 계산하는 기준
#     - categorical_crossentropy : 여러 후보 중 하나를 고르는 문제(다중 분류)에 쓰는 손실 함수
#       → 매 자리마다 "수천 개 단어/글자 중 어떤 게 다음에 올까"를 맞히는 문제이므로 이걸 쓰겠다
#       → 정답이 원-핫 인코딩(정답 위치만 1, 나머지는 0인 벡터) 형태일 때 사용함

In [41]:
# ③ 학습 실행


model.fit([encoder_input_data, decoder_input_data], decoder_target_data,
          batch_size=batch_size, epochs=epochs, validation_split=0.2)


# → fit : 실제 데이터를 넣어 학습을 시작하겠다
# → [encoder_input_data, decoder_input_data] : ①에서 정한 입구 2개에 순서대로 데이터를 넣겠다
#     - encoder_input_data : 영어 문장들
#     - decoder_input_data : 한국어 정답 문장들 (보통 맨 앞에 "시작" 표시 토큰이 붙어 있음)
# → decoder_target_data : 모델이 맞혀야 할 정답
#     - decoder_input_data를 한 칸 앞으로 당긴 문장 (시작 토큰 없이, 끝에 "종료" 토큰이 붙음)
#     - 예) 입력 : [시작, 나는, 학생, 이다]  →  정답 : [나는, 학생, 이다, 종료]
#       → "지금 단어를 보고 그다음 단어를 맞혀라"를 학습시키겠다
# → batch_size=batch_size : 데이터를 한 번에 몇 개씩 묶어서 학습할지 정하겠다 (앞에서 변수로 정해둔 값 사용)
#     - 전체를 한꺼번에 넣으면 메모리가 부족하고, 1개씩 넣으면 너무 느려서 적당히 묶음
# → epochs=epochs : 전체 데이터를 처음부터 끝까지 몇 번 반복해서 학습할지 정하겠다
#     - 실행 결과의 "Epoch 1/30"은 30번 중 1번째 반복 중이라는 뜻
# → validation_split=0.2 : 데이터의 20%는 학습에 쓰지 않고 떼어두고, 매 epoch마다 성능 확인용(검증 데이터)으로 쓰겠다
#     - 모델이 학습 데이터만 외워버리는 과적합(overfitting)이 일어나는지 확인하기 위함

Epoch 1/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 12s 90ms/step - loss: 1.5416 - val_loss: 2.1344
Epoch 2/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - loss: 1.5373 - val_loss: 2.1293
Epoch 3/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 67ms/step - loss: 1.5338 - val_loss: 2.1301
Epoch 4/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - loss: 1.5311 - val_loss: 2.1293
Epoch 5/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 84ms/step - loss: 1.5282 - val_loss: 2.1203
Epoch 6/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 94ms/step - loss: 1.5251 - val_loss: 2.1186
Epoch 7/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - loss: 1.5213 - val_loss: 2.1176
Epoch 8/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 59ms/step - loss: 1.5192 - val_loss: 2.1125
Epoch 9/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 57ms/step - loss: 1.5153 - val_loss: 2.1078
Epoch 10/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 55ms/step - loss: 1.5092 - val_loss: 2.1053
Epoch 11/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 55ms/step - loss: 1.5048 - val_loss: 2.0967
Epoch 12/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 57ms/step - loss: 1.

In [42]:
# ④ 마지막 주석 해석


# Encoder 학습 + Decoder Teacher Focing


# → 이 학습 과정에서 인코더는 영어 문장의 의미를 요약하는 법을 배우겠다
# → 디코더는 Teacher Forcing(교사 강요) 방식으로 학습하겠다 (주석의 Focing은 Forcing의 오타)
#     - Teacher Forcing : 디코더가 다음 단어를 예측할 때, 자기가 직전에 예측한 단어 대신 "실제 정답 단어"를 입력으로 넣어주는 방식
#     - 이유 : 학습 초반엔 예측이 엉망이라, 틀린 단어를 계속 다음 입력으로 쓰면 오류가 쌓여 학습이 잘 안 됨
#       → 정답을 떠먹여 주면 학습이 빠르고 안정적임
#     - 그래서 ③에서 decoder_input_data로 정답 문장을 직접 넣어준 것

In [43]:
# Decoder의 추론 (Autoregressive) : 학습이 끝난 층을 재사용해 추론용 모델 2개를 조립
#   정답이 없으므로 한 글자씩 예측 → 그 글자와 기억(h, c)을 다음 입력으로 넘기는 구조가 필요


# ── 추론용 인코더 모델 ──
# Model(encoder_input, encoder_state) : 새로운 영어 문장이 등장했을 때, 해당 정보를 처리하는 Encoder
#   입력 → encoder_input (영어 원-핫 문장)
#   출력 → encoder_state ([state_h, state_c] = 컨텍스트 벡터)
#   학습 때 만든 encoder 층(LSTM)을 그대로 이어 쓰므로, 학습된 가중치도 그대로 사용 (새로 배우지 않음)
encoder_model = Model(encoder_input, encoder_state)

# ── 추론용 디코더 모델 ──
# Decoder LSTM Layer에 초기값을 지정하기 위한 입구 2개
#   Input(shape=(node_num,)) → 숫자 64개짜리 벡터 하나를 받는 입구 (Decoder Node의 초기값)
#   첫 글자 때는 컨텍스트 벡터, 그다음부터는 직전 글자를 처리한 뒤의 기억이 들어오는 자리
#   학습 땐 encoder_state를 한 번만 넣었지만, 추론 땐 매 글자마다 "직전 기억"을 바꿔 넣어야 해서 입구로 만듦
decoder_state_input_h = Input(shape=(node_num,))   # 직전 시점의 단기 기억 h
decoder_state_input_c = Input(shape=(node_num,))   # 직전 시점의 장기 기억 c

# 두 입구를 리스트로 묶어 LSTM의 initial_state 형식에 맞춤
decoder_state_inputs = [decoder_state_input_h, decoder_state_input_c]

# 구성된 Input 정보를 이용하여 Decoder Layer 구성 (학습 때 만든 decoder 층을 재사용 → 가중치 공유)
#   decoder_input                        → 이번에 넣을 한국어 글자 (추론 땐 한 글자씩)
#   initial_state=decoder_state_inputs   → 컨텍스트 벡터(첫 글자) 또는 직전 글자의 기억(그다음부터)
#   state_h, state_c                     → 이번 글자를 처리한 뒤의 새 기억 → 다음 글자로 넘길 값
decoder_output, state_h, state_c = decoder(decoder_input, initial_state=decoder_state_inputs)

# 다음 Step으로 전달될 상태를 지정 → 새 기억을 리스트로 묶어 모델 출력으로 내보내고, 다음 반복에 다시 넣음
decoder_state = [state_h, state_c]

# Softmax 함수에 의해 출력 값이 도출 (학습 때 만든 decoder_dense 층을 재사용 → 64개 숫자를 802개 글자 확률로 변환)
decoder_outputs = decoder_dense(decoder_output)

# 다음 단계의 토큰(글자)을 예측하고, 다음 글자를 생성하기 위한 새로운 상태를 구성 (Autoregressive)
# Model(입력, 출력) : 추론용 디코더 모델 조립
#   입력 → [decoder_input] + decoder_state_inputs = [글자, h, c] 3개
#   출력 → [decoder_outputs] + decoder_state      = [글자 확률, 새 h, 새 c] 3개
#   리스트끼리 + 로 이어 붙여 입력·출력을 각각 3개짜리 목록으로 만든 것
decoder_model = Model([decoder_input] + decoder_state_inputs,
                      [decoder_outputs] + decoder_state)


# → 학습된 인코더 층으로 "영어 문장 → 컨텍스트 벡터"만 계산하는 encoder_model을 만들겠다
# → 디코더에 직전 기억(h, c)을 직접 넣을 수 있도록 입구 2개를 새로 만들겠다
# → 학습된 decoder, decoder_dense 층을 재사용해 "글자 + 기억 → 다음 글자 확률 + 새 기억"을 계산하게 연결하겠다
# → 입력 3개(글자, h, c) → 출력 3개(확률, 새 h, 새 c)인 decoder_model을 만들겠다
# ※ 새 층을 만들지 않고 학습된 층을 다시 연결만 했으므로, 학습한 내용(가중치)이 그대로 쓰임 = 가중치 공유

In [44]:
# 디코더에 의해 계산된 숫자 결과를 다시 문자로 변환하는 작업을 수행 
# 번호 → 글자 역방향 사전 생성 (Num2Eng / Num2Kor)
#   모델은 글자가 아니라 "번호(확률이 가장 큰 칸의 위치)"를 내놓으므로, 그 번호를 다시 글자로 바꿀 변환표가 필요


# input_token_index.items()  → {글자: 번호} 사전에서 (글자, 번호) 짝을 차례로 꺼냄 → (' ', 0), ('!', 1), ...
# for char, i in ...         → char = 글자, i = 번호
# (i, char)                  → 순서를 뒤집어 (번호, 글자)로 만듦 → 번호를 넣으면 글자가 나오게
# dict([...])                → (키, 값) 짝 목록을 사전으로 변환 → {0: ' ', 1: '!', ...}
reverse_input_char_index = dict([(i, char) for char, i in input_token_index.items()])

# 한국어도 같은 방식 → {0: '\t', 1: '\n', 2: ' ', ...}
#   번역 결과를 글자로 되돌릴 때 실제로 쓰는 건 이 한국어 역방향 사전
reverse_target_char_index = dict([(i, char) for char, i in target_token_index.items()])


# → {글자: 번호} 사전을 뒤집어 {번호: 글자} 사전을 만들겠다 (영어 / 한국어 각각)
# → 디코더가 내놓은 번호(argmax 결과)를 한국어 글자로 되돌리는 데 쓰겠다
# ※ 앞의 token_index 셀은 enumerate의 (번호, 글자)를 뒤집었고, 여기선 items()의 (글자, 번호)를 뒤집은 것

In [45]:
# decode_sequence : 영어 문장(원-핫) 하나를 받아 한국어 문장을 한 글자씩 생성하는 함수 (Autoregressive)
# 입력 문장이 시퀀스 형태로 들어왔을 때, 해당 문장을 번역하는 함수를 구성
#   흐름 : 인코더로 컨텍스트 벡터 계산 → \t로 시작 → 글자 예측 → 그 글자와 새 기억을 다음 입력으로 → \n이면 멈춤


# def 함수이름(매개변수): → 함수 정의 / input_seq = 번역할 영어 문장 1개의 원-핫 배열 (1, 글자 수, 67)
def decode_sequence(input_seq):

    # ① 입력받은 Sequence를 이용해서, Encoder에서 Context Vector 생성
    #   encoder_model.predict(...) → 학습된 인코더에 영어 문장을 넣어 [h, c]를 계산 (학습 X, 계산만)
    #   verbose=0                  → 진행 표시줄을 출력하지 않음 (글자마다 찍히면 화면이 지저분해짐)
    #   state_value                → [h, c] = 컨텍스트 벡터 → 디코더 첫 글자의 초기 기억
    state_value = encoder_model.predict(input_seq, verbose=0)

    # ② 추론 문장 구성 : 모든 토큰 값이 0으로 구성된 Matrix를 선언
    #   np.zeros((1, 1, 802)) → (문장 1개, 글자 1개, 802칸) 모양의 빈 원-핫 배열
    target_seq = np.zeros((1, 1, len(target_characters)))

    # SOS(Start Of Sequence) 토큰을 문장 맨 앞에 입력 → Decoder가 추론을 시작하는 부분을 지정
    #   target_token_index['\t'] → '\t'의 번호(0번) → 그 칸만 1로 칠함 = "생성 시작" 신호
    target_seq[0, 0, target_token_index['\t']] = 1

    # ③ Decoding 실시 (추론 수행) : for가 아니라 while을 쓰는 이유
    #   for문은 "몇 번 반복할지"를 미리 정해야 함
    #   그런데 번역 결과의 길이는 미리 알 수 없음 → Hi(2글자) → 안녕하세요(5글자)처럼 입력과 출력 길이가 다름
    #   그래서 "조건이 참인 동안 계속"하는 while로 한 글자씩 만들다가, 아래 종료 조건 중 하나가 되면 멈춤
    #     1. EOS 토큰('\n')이 생성되면, 생성 모델의 작동을 중지 → 모델이 스스로 "문장 끝"을 정함
    #     2. 사용자가 지정한 문장의 길이(최대 길이)를 벗어나는 경우, 생성 모델의 작동을 중지 → 무한 반복 방지
    stop_cond = True        # 추론을 진행할지 중지할지 판별하는 변수 (True → 추론 / False → 정지)
    decoder_sentence = ''   # 생성된 문자를 받아줄 변수 (빈 문자열에서 시작해 글자를 이어 붙임)

    # ④ 위의 종료 조건이 발동될 때까지 반복
    while stop_cond:

        # 디코더의 Layer 정보를 이용해 Output Token 생성 (추론)
        #   [target_seq] + state_value → [지금 글자, 직전 h, 직전 c] 3개 입력 (리스트끼리 + 로 이어 붙임)
        #   output_token               → 다음 글자 확률 (1, 1, 802)
        #   h, c                       → 이번 글자를 처리한 뒤의 새 기억
        output_token, h, c = decoder_model.predict([target_seq] + state_value, verbose=0)

        # 가장 확률이 높게 계산된 숫자값을 확인 → 가장 확률이 높게 나온 한글의 위치 정보(번호)
        #   output_token[0, -1, :] → 0번 문장, 마지막 시점(-1), 802칸 전체
        #   np.argmax(...)         → 값이 가장 큰 칸의 번호 (예: 250)
        sampled_token_index = np.argmax(output_token[0, -1, :])

        # 앞서 찾은 가장 높은 확률의 글자를 한국어로 변환 (역방향 사전 : 250 → '들')
        sampled_char = reverse_target_char_index[sampled_token_index]

        # 생성한 글자를 문장에 이어 붙임 ('' → '들' → '들어' → ...)
        decoder_sentence += sampled_char

        # 종료 조건이 실행되는 파트
        #   sampled_char == '\n' → EOS에 의한 종료 (End Of Sequence가 등장하면 추론을 멈춤)
        #   len(decoder_sentence) >= max_decoder_seq_length → 학습한 문장의 최대 길이(30)를 넘어가면 종료
        #     ('\n'을 끝내 못 만들고 계속 글자를 뽑는 경우를 막는 안전장치)
        if (sampled_char == '\n' or len(decoder_sentence) >= max_decoder_seq_length):
            stop_cond = False

        # 추론이 종료되지 않았다면, 다음 글자 예측을 위한 변수를 초기화 (Autoregressive의 핵심)
        #   새 빈 배열을 만들고, 방금 고른 글자의 번호 칸만 1로 칠함 → 방금 예측한 글자가 다음 입력이 됨
        target_seq = np.zeros((1, 1, len(target_characters)))
        target_seq[0, 0, sampled_token_index] = 1

        # 새 기억(h, c)을 다음 반복의 초기 기억으로 넘김
        state_value = [h, c]

    # 종료 조건에 의해 반복이 멈추면, 생성된 문장을 출력 (끝에 '\n'이 포함되어 있음)
    return decoder_sentence


# → 인코더로 영어 문장의 컨텍스트 벡터를 계산하겠다
# → '\t'(SOS) 한 글자로 디코더를 시작하겠다
# → 디코더가 내놓은 802개 확률 중 가장 큰 글자를 골라 문장에 이어 붙이겠다
# → 방금 고른 글자와 새 기억(h, c)을 다음 입력으로 넣어 반복하겠다 (Autoregressive)
# → '\n'(EOS)이 나오거나 최대 길이에 도달하면 멈추고, 완성된 문장을 돌려주겠다
# ※ 이 셀은 함수를 "정의"만 함 → 실행해도 출력 없음, 다음 셀에서 호출해야 번역이 실행됨

- decode_sequence 핵심 포인트
    - **while을 쓰는 이유**
        - for : "10번 반복해"처럼 반복 횟수를 **미리 알 때** 사용
        - while : "조건이 맞는 동안 계속해"처럼 **끝나는 시점을 모를 때** 사용
        - 번역 결과는 몇 글자가 될지 돌려 보기 전엔 모름 (Hi → 안녕하세요, 2글자 → 5글자) → while
        - 단, while은 조건이 영원히 참이면 끝나지 않는 위험 → **최대 길이**를 두 번째 종료 조건(안전장치)으로 함께 사용
    - **종료 조건 2가지**
        1. EOS('\n')가 생성되면 중지 → 모델이 스스로 문장 끝을 정함
        2. 문장 길이가 최대 길이(max_decoder_seq_length = 30)에 도달하면 중지 → 무한 반복 방지
    - **들여쓰기 주의**
        - `state_value = [h, c]` → while **안쪽** (8칸) : 매 글자마다 새 기억을 다음 반복으로 넘김
        - `return decoder_sentence` → while **바깥**, 함수 안쪽 (4칸)
        - return이 while 안으로 들어가면 첫 글자 하나만 만들고 함수가 끝나 버림

In [46]:
# 번역 테스트 : 학습 데이터 중 50~54번 문장 5개를 번역해 결과 확인
# 번역 txt 파일에 있는 50번째부터 54번째까지 영어 문장을 넣어 번역이 잘 수행되는지 확인
#   ※ 파이썬은 0부터 세므로, 인덱스 50번 = 파일의 51번째 줄 (Be calm. / Be fair. / Beat it. / Call us. / Come in.)


# range(50, 55) → 50, 51, 52, 53, 54 (끝 숫자 55는 포함하지 않음) → 문장 5개
for seq_index in range(50, 55):

    # 원-핫 배열에서 seq_index번 문장 1개만 잘라냄
    #   [seq_index:seq_index+1] → 슬라이싱으로 자르면 모양이 (1, 26, 67)로 3차원 유지
    #   [seq_index]만 쓰면 (26, 67) 2차원이 되어, 모델이 기대하는 (배치, 글자 수, 67) 모양과 안 맞아 에러
    input_seq = encoder_input_data[seq_index:seq_index+1]

    # 앞에서 만든 함수에 넣어 한국어 문장을 한 글자씩 생성 (Autoregressive)
    result = decode_sequence(input_seq)

    # 원래 영어 문장과 번역 결과를 나란히 출력
    #   input_text[seq_index] → 원-핫이 아닌 원래 영어 문장 (사람이 읽을 수 있는 형태)
    print('입력 문장 :', input_text[seq_index])
    print('출력 문장 :', result)     # result 끝에 '\n'이 있어서 한 줄이 더 띄워져 보임
    print('-------------------------')


# → 50번부터 54번까지 5개 문장을 하나씩 꺼내겠다
# → 각 문장을 3차원 모양 그대로 잘라 decode_sequence에 넣어 번역하겠다
# → 원래 영어 문장과 생성된 한국어 문장을 나란히 출력하겠다
# ※ 50~54번은 학습에 쓴 문장(앞쪽 80%) → 처음 보는 문장으로 확인하려면 range(2500, 2505)처럼 뒤쪽 20%를 사용

입력 문장 : Be calm.
출력 문장 : 그 이 .

-------------------------
입력 문장 : Be fair.
출력 문장 : 그 이 어.

-------------------------
입력 문장 : Beat it.
출력 문장 : 그 이.

-------------------------
입력 문장 : Call us.
출력 문장 : 그 이 .

-------------------------
입력 문장 : Come in.
출력 문장 : 우리 어.

-------------------------


- Seq2Seq 모형의 한계점 :
    1. 긴 Sequence에 대해 RNN 계열 모델의 특성으로 기울기 소실 / 폭주 문제가 발생
        - ※ 역전파 신호가 시점을 거슬러 갈 때마다 곱해짐 → 1보다 작으면 점점 사라지고(소실), 크면 점점 커짐(폭주)
        - ※ 결과적으로 문장 앞부분 글자는 학습 신호를 거의 받지 못함 (LSTM의 cell state가 완화하지만 긴 문장에선 여전히 한계)
    2. 고정 크기의 Context Vector에 입력 문장의 모든 정보를 압축하므로 정보 손실(병목)이 발생
        - ※ 8글자든 26글자든 똑같이 [h, c] = 숫자 128개로 압축 → 문장이 길수록 앞부분 정보가 흐려짐
        - ※ 디코더는 이 요약 하나만 보고 전체 문장을 생성해야 함

- 이 한계점을 보완하기 위해 만들어진 것이 **Attention Mechanism**
    - 디코더가 글자를 만들 때마다 인코더의 **모든 시점 출력**을 다시 보고, 지금 필요한 위치에 가중치를 줌
    - 매 시점 새 컨텍스트가 만들어짐 → 고정 크기 병목(2번) 해소
    - 먼 거리의 글자도 한 번에 연결됨 → 기울기 소실(1번) 완화

# 시뮬레이터 HTML을 기본 브라우저로 열기


import os

# os.startfile(경로) : 파일을 더블클릭한 것처럼 기본 프로그램(브라우저)으로 엶 (윈도우 전용)
#   '../lecture_materials/...' → my_code에서 한 칸 위(Day10)로 올라가 lecture_materials 폴더의 파일
os.startfile(os.path.abspath('../lecture_materials/Seq2Seq_Attention_시뮬레이터.html'))


# → lecture_materials 폴더의 시뮬레이터를 브라우저로 열겠다

In [47]:
### Seq2Seq · Attention 시뮬레이터

- 위치 : `C:\Users\Taeeun Eom\Desktop\TEN\11_Bootcamp\12_NLP\Day10\lecture_materials\Seq2Seq_Attention_시뮬레이터.html`
- 오늘 노트북에서 한 **데이터 준비 → 학습 → 한 글자씩 번역**을 브라우저 안에서 눈으로 따라가 보는 도구
- 화면이 ①②③ 세 구역으로 나뉘어 있으니 순서대로 보기

---

**① 데이터**

- `영어 | 한국어` 형태의 문장 쌍 → 노트북의 `kor.txt`를 아주 작게 줄인 버전
- 전처리 규칙이 노트북과 동일
    - 디코더 입력 : `SOS 들 어 와 .` (노트북의 `\t`)
    - 디코더 정답 : `들 어 와 . EOS` (노트북의 `\n`) → **한 칸 밀림**
    - **마스크** : 노트북과 다른 점. 패딩으로 채운 0 칸을 **계산에서 아예 빼 주는** 장치 (노트북은 이 처리가 없어 패딩 칸도 계산에 섞임)

**② 학습 — Teacher Forcing**

1. 은닉 크기(= 노트북의 `node_num`)와 epoch를 고르고 **"Seq2Seq 학습"** 클릭 (기본값 기준 1~2분)
2. 끝나면 **"Attention 학습"**도 클릭해 두 모델을 모두 만들기
3. 아래 표에서 **입력 / 정답 / Seq2Seq 생성 / Attention 생성** 비교 → 생성 열은 **자기회귀** 결과라 두 모델의 실력 차이가 보임

- "손실이 0.05 아래면 대부분 외운 상태" = 오늘 배운 **과적합**

**③ 시뮬레이터 — 한 시점씩 따라가기 (가장 중요)**

1. 모델 **Seq2Seq**, 디코더 입력 **Teacher Forcing**으로 두고 문장을 골라 **실행**
2. **"다음 ▶"**을 한 번씩 누르며 시점 이동 → 오른쪽 **"현재 단계 내부 값"**에 원-핫 번호, **h·c 값**, **Softmax 후보**(한국어 글자 확률 중 상위 글자들) 표시 → 노트북의 `np.argmax`가 **어떤 후보를 골랐는지** 확인
3. 디코더 입력을 **자기회귀**로 바꿔 같은 문장 실행 → **후보 글자를 클릭해 일부러 틀리게** 만들어 보기
    - 틀린 글자가 다음 입력이 되며 **뒤 글자까지 줄줄이 흔들림 = 노출 편향**
    - Teacher Forcing에서는 같은 곳을 틀려도 다음 입력은 정답이라 흔들리지 않음
4. 모델을 **Attention**으로 바꿔 실행 → 맨 아래 **Attention 가중치 α 표** 등장
    - 한국어 글자를 만들 때마다 **영어의 어느 글자를 보고 있는지** 색으로 표시
    - 예) `들`을 만들 땐 `Come` 쪽, `와`를 만들 땐 `in` 쪽이 진한지 확인
    - 한계점 정리의 **"필요한 위치에 가중치를 준다"**의 실제 모습

---

**보면서 확인할 질문 두 가지**

- **긴 문장**으로 Seq2Seq와 Attention 비교 → **병목** 때문에 Seq2Seq는 긴 문장 앞부분을 틀리기 쉬움
- Attention α 표에 **대각선 모양**이 보이면 영어와 한국어의 **글자 순서를 대응**시키는 중 → 어순이 달라 완벽한 대각선이 아니라 **구부러진 모양**

SyntaxError: invalid syntax (4039534516.py, line 3)